## ECCO-TCP Passage Metadata and Topic Modeling

### Overview

This notebook:

1. Builds a **passage-level metadata file** for 1000-word ECCO-TCP passages stored as `.txt` files in `complete_ecco_tcp_passages/`.
2. Joins each passage to its **volume-level metadata** from `ECCOTCP.csv` (columns: `TCP`, `Author`, `Date`, `Title`, `Pages`, etc.).
3. Preprocesses passage texts and builds a **tf–idf representation**.
4. Trains an **NMF topic model** with `k = 80` topics.
5. Assigns each passage a **primary topic** and a **confidence score**; passages with confidence `< 0.3` are assigned to the `OTHER` category.

The output is a passage-level CSV with metadata and topic assignments, suitable for later use in conceptual diversity metrics.

### 1. Imports and Configuration

In [ ]:
import os
import glob
import pandas as pd
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# Directory containing 1000-word passage .txt files
PASSAGE_DIR = "complete_ecco_tcp_passages"

# Volume-level metadata CSV (original ECCO-TCP)
ECCO_TCP_CSV = "ECCOTCP.csv"

# Output CSV for passage-level metadata + topic assignments
OUTPUT_PASSAGE_CSV = "ecco_tcp_passages_with_topics.csv"

# Topic modeling parameters
N_TOPICS = 80
TOPIC_CONFIDENCE_THRESHOLD = 0.3

# TF-IDF parameters (you can tune these)
MAX_FEATURES = 50000     # limit vocabulary size for memory
MIN_DF = 5              # ignore very rare terms

### 2. Load ECCO-TCP Volume-level Metadata

We load `ECCOTCP.csv`, which contains columns such as:

- `TCP` (volume identifier, e.g., "K000039")
- `Author`, `Date`, `Title`, `Pages`, etc.

We will later join this to passages by matching the `TCP` ID extracted from passage filenames (e.g., `"K000039"` in `"K000039.000.00001.txt"`).

In [ ]:
ecco_meta = pd.read_csv(ECCO_TCP_CSV)

print("Volume-level metadata columns:", list(ecco_meta.columns))
print("Number of volumes in ECCO_TCP metadata:", len(ecco_meta))

In [ ]:
# We expect a column named 'TCP' for the volume identifier.
assert "TCP" in ecco_meta.columns, "Expected a 'TCP' column in ECCOTCP.csv"

### 3. Build passage-level metadata frame

In [ ]:
# Find all .txt files in the passage directory
passage_files = sorted(glob.glob(os.path.join(PASSAGE_DIR, "*.txt")))
print(f"Found {len(passage_files)} passage files.")

# Build a DataFrame with passage_id, tcp_id, file_path
passage_records = []

for fpath in tqdm(passage_files):
    fname = os.path.basename(fpath)
    # Strip extension
    base = os.path.splitext(fname)[0]  # e.g., "K000039.000.00001"
    # TCP id assumed to be before the second dot
    tcp_id = base.rsplit(".", 1)[0]        # e.g., "K000039.000"
    
    passage_records.append({
        "passage_id": base,
        "tcp_id": tcp_id,
        "file_path": fpath,
    })

passages_df = pd.DataFrame(passage_records)
print("Passage-level frame shape:", passages_df.shape)
passages_df.head()

### 4. Join with volume-level metadata

In [ ]:
# Join passage records with volume metadata
# left_on 'tcp_id' (from filenames), right_on 'TCP' (from ECCOTCP.csv)
passages_df = passages_df.merge(
    ecco_meta,
    left_on="tcp_id",
    right_on="TCP",
    how="left"
)

print("After join, shape:", passages_df.shape)

# Check how many passages failed to find metadata
missing_meta = passages_df["Author"].isna().sum()
print(f"Passages with missing volume metadata: {missing_meta}")

# Inspect a few rows
passages_df.head()

### 5. Load passage texts

We now read the content of each 1000-word passage `.txt` file into a `text` column in `passages_df`.

Depending on your machine, loading 150k passages into memory is substantial but typically feasible. If you run into memory issues, we can later adapt this to process in chunks.

In [ ]:
def read_text_safe(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

print("Loading passage texts...")

texts = []
for fpath in tqdm(passages_df["file_path"].tolist()):
    texts.append(read_text_safe(fpath))

passages_df["text"] = texts

print("Example passage text snippet:")
print(passages_df["text"].iloc[0][:500])

### 6. Build custom stopwords list

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Start from scikit-learn's built-in English stopword list
custom_stopwords = set(ENGLISH_STOP_WORDS)

# Domain-specific structural / paratext stopwords for ECCO-TCP
# (all lowercased, because TfidfVectorizer lowercases by default)
STRUCTURAL_STOPWORDS = {
    # Generic structural words
    "page", "pages", "chap", "chapter", "sect", "section",
    "vol", "volume", "tom", "tome", "part", "parts",
    "pl", "plate", "plates", "fig", "figure", "tab", "table",
    "contents", "index", "appendix", "preface", "introduction",
    "hist", "edit", "edition",

    # Reference / footnote shorthand
    "ibid", "ib", "fol", "seq", "mss", "supr", "supra",
    "loc", "cit", "viz",

    # Errata / printer marks
    "errata", "dele", "numb", "num", "no", "nos",

    # Subscription / prosopography titles (optional)
    "esq", "esquire", "bart", "hon", "rev", "dr",

    # Short Bible/book abbreviations that mostly indicate verse refs
    "ps", "cor", "mat",

    # Numbers and labeling
    "ii", "iii", "iv", "vi", "vii", "viii", "ix", "xi", "xii", "10", 
    "11", "12", "13", "14", "16", "15", "17", "18", "19", "20", "22", 
    "21", "cor", "23", "24", "25", "26",

    # Foreign languages
    "et", "ad", "non", "est", "quod", "cum", "ut", "vel", "pro", "qui", 
    "ac", "si", "aut", "quae", "quam", "nec", "ex", "sed", "hoc", "nobis",
    "la", "le", "les", "que", "il", "qu", "des", "en", "je", "et", "dans", 
    "du", "ne", "qui", "vous", "ce", "une", "motte", "pour", "est",

    # Additional words
    "thing", "things", "ul", "use", "ib", "110", "160", "181"
}

custom_stopwords |= STRUCTURAL_STOPWORDS

print(f"Total custom stopwords: {len(custom_stopwords)}")
list(sorted(list(STRUCTURAL_STOPWORDS)))[:20]

### 6. Build tf–idf representation

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    max_df=0.8,
    stop_words=list(custom_stopwords) 
)

print("Fitting TF–IDF vectorizer and transforming passages (this may take a while)...")
tfidf_matrix = tfidf_vectorizer.fit_transform(passages_df["text"])

print("TF–IDF matrix shape:", tfidf_matrix.shape)

### 7. Train NMF topic model (k = 80)

In [ ]:
nmf_model = NMF(
    n_components=N_TOPICS,
    init="nndsvda",
    random_state=0,
    max_iter=200
)

print("Fitting NMF model (this may take some time)...")
W = nmf_model.fit_transform(tfidf_matrix)  # shape: (n_passages, N_TOPICS)
H = nmf_model.components_                  # shape: (N_TOPICS, n_terms)

print("Document-topic matrix W shape:", W.shape)
print("Topic-term matrix H shape:", H.shape)

### 8. Recompute topic proportions and labels

In [ ]:
import numpy as np

# W: document-topic matrix, shape (n_docs, N_TOPICS)
# Normalize each row to sum to 1 to get topic proportions.
row_sums = W.sum(axis=1, keepdims=True)

# Avoid division by zero for any all-zero rows (rare, but safe)
row_sums[row_sums == 0] = 1.0

doc_topic_props = W / row_sums  # shape: (n_docs, N_TOPICS), rows sum to 1

# Primary topic and (normalized) confidence
primary_topics = np.argmax(doc_topic_props, axis=1)
confidences = doc_topic_props.max(axis=1)

passages_df["primary_topic_id"] = primary_topics
passages_df["topic_confidence"] = confidences

# Apply threshold τ on *proportions*, not raw W
tau = 0.2  # now this is realistic
def topic_label(row):
    if row["topic_confidence"] < tau:
        return "OTHER"
    else:
        return f"TOPIC_{int(row['primary_topic_id'])}"

passages_df["topic_label"] = passages_df.apply(topic_label, axis=1)

print("Topic label counts (including OTHER):")
print(passages_df["topic_label"].value_counts().head(20))

print("\nProportion in OTHER:", 
      (passages_df["topic_label"] == "OTHER").mean())

In [ ]:
passages_df["topic_label"] = passages_df.apply(topic_label, axis=1)

passages_df[["passage_id", "tcp_id", "primary_topic_id", "topic_confidence", "topic_label"]].head()

### 9. Extract and inspect topics

In [ ]:
# Number of top words per topic to display
N_TOP_WORDS = 20

# Get the vocabulary (list of term strings)
feature_names = tfidf_vectorizer.get_feature_names_out()

topics_output_path = "nmf_topics_k80.txt"

with open(topics_output_path, "w", encoding="utf-8") as f:
    for topic_idx, topic in enumerate(nmf_model.components_):
        # topic is an array of shape (n_terms,) with weights
        top_indices = topic.argsort()[::-1][:N_TOP_WORDS]
        top_terms = [feature_names[i] for i in top_indices]
        
        f.write(f"Topic {topic_idx}\n")
        f.write(", ".join(top_terms) + "\n\n")

print(f"Wrote topics to: {topics_output_path}")

### 9. Save CSV

In [ ]:
# Save passage-level metadata with topics, dropping the column that contains the text of all passages
passages_df.drop(columns=["text"]).to_csv(OUTPUT_PASSAGE_CSV, index=False)
print(f"Saved passage-level metadata (without text) to: {OUTPUT_PASSAGE_CSV}")